In [1]:
from langchain_core.documents import Document

In [4]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
import os

HF_TOKEN = os.environ["HUGGINGFACEHUB_API_TOKEN"]

In [5]:
documents = [
    Document(page_content="Langchain helps developers build LLM aplications easily"),
    Document(page_content="Chroma is a vector database optimized for LLM-based search"),
    Document(page_content="Embeddings convert text into high-dimensional vectors"),
    Document(page_content="OpenAI provides powerful embedding models")
]

In [6]:
embedding_model = HuggingFaceEmbeddings(model_name="BAAI/bge-base-en-v1.5",  model_kwargs={"token": HF_TOKEN})

vector_store = Chroma.from_documents(
    documents = documents,
    embedding= embedding_model,
    collection_name="my_collection"
)

c:\Users\Hannan\Desktop\things\agentic\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1788.45it/s]


In [7]:
retreiver = vector_store.as_retriever(search_kwargs={"k":2})

In [8]:
query = "What is Chroma used for?"
results = retreiver.invoke(query)

In [9]:
for i, doc in enumerate(results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Chroma is a vector database optimized for LLM-based search

--- Result 2 ---
Langchain helps developers build LLM aplications easily


In [10]:
docs1 = [
    Document(page_content = "Langchain makes it easy to work with LLMs"),
    Document(page_content = "Langchain is used to build LLM based applications."),
    Document(page_content = "Chroma is used to store and search document embeddings"),
    Document(page_content = "Embeddings are vector representations of text"),
    Document(page_content = "MMR help to fetch diverse results when doing similarity search."),
    Document(page_content = "Langchain supports Chroma, FAISS, Pinecone, and more") 
]

In [11]:
from langchain_community.vectorstores import FAISS

vector_store1 = FAISS.from_documents(
    documents=docs1,
    embedding=embedding_model
)

In [12]:
retreiver1 = vector_store1.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "lambda_mult": 0.5}
)

In [13]:
query = "What is langchain ?"
results1 = retreiver1.invoke(query)

In [14]:
for i,doc in enumerate(results1):
    print(f"--- Result {i+1} ---")
    print(doc.page_content)

--- Result 1 ---
Langchain is used to build LLM based applications.
--- Result 2 ---
Langchain supports Chroma, FAISS, Pinecone, and more
--- Result 3 ---
Embeddings are vector representations of text


In [19]:
from langchain_classic.retrievers import MultiQueryRetriever

In [20]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [21]:
vector_store2 = FAISS.from_documents(documents = all_docs, embedding=embedding_model,  )

In [22]:
similarity_retriever = vector_store2.as_retriever(search_type="similarity", search_kwargs={"k":5})

In [41]:
from langchain_huggingface import HuggingFaceEndpoint, ChatHuggingFace

llm = HuggingFaceEndpoint(
    repo_id= "deepseek-ai/DeepSeek-V4.1-Flash",
    task= "text-generation"
)

model = ChatHuggingFace(llm=llm)

In [30]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vector_store2.as_retriever(search_kwargs={"k": 5}),
    llm=model
)

In [32]:
query = "How to improve energy levels and maintain balance?"

In [33]:
# Retrieve results
similarity_results = similarity_retriever.invoke(query)
multiquery_results= multiquery_retriever.invoke(query)

In [34]:
for i, doc in enumerate(similarity_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)

print("*"*150)

for i, doc in enumerate(multiquery_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 3 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.

--- Result 4 ---
Regular walking boosts heart health and can reduce symptoms of depression.

--- Result 5 ---
Deep sleep is crucial for cellular repair and emotional regulation.
******************************************************************************************************************************************************

--- Result 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.

--- Result 2 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.

--- Result 3 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.

--- Result 4 ---
Regular walking boosts heart health and can reduce symptoms of

In [42]:
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_classic.retrievers.document_compressors import LLMChainExtractor

In [43]:
# Recreate the document objects from the previous data
docs3 = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [44]:
vector_store3 = FAISS.from_documents(documents=docs3, embedding=embedding_model)

In [45]:
base_retriever = vector_store3.as_retriever(search_kwargs={"k":5})

In [54]:
llm1 = HuggingFaceEndpoint(
    repo_id = "meta-llama/Llama-3.1-8B-Instruct",
    task= "text-generation"
)

model1 = ChatHuggingFace(llm=llm1)

compressor = LLMChainExtractor.from_llm(model1)

In [55]:
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

In [56]:
query = "What is Photosynthesis ?"
compressed_results = compression_retriever.invoke(query)

In [57]:
for i, doc in enumerate (compressed_results):
    print(f"\n--- Result {i+1} ---")
    print(doc.page_content)


--- Result 1 ---
> Photosynthesis is the process by which green plants convert sunlight into energy.

--- Result 2 ---
The chlorophyll in plant cells captures sunlight during photosynthesis.
